In [1]:
import os
import tqdm
import requests
import typing as tt

import sys
from pathlib import Path
import pandas as pd
import numpy as np

import data
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.feather as pf

In [2]:
dataset = data.get_dataset()
feats_v1 = data.get_features_v1(dataset)

In [5]:
osrm_table = pf.read_table("osrm-v1.feather")
osrm_table.schema

idx: int64
route_distance: double
route_duration: double

In [6]:
print(feats_v1.shape)
print(osrm_table.shape)

(54273658, 27)
(54293690, 3)


In [7]:
osrm_dataset = ds.dataset(osrm_table)
osrm_table2 = osrm_dataset.to_table(
    columns={
        "idx": pc.field('idx').cast(pa.uint32()),
        "route_distance": (pc.field("route_distance") / 1000.0).cast(pa.float32()),
        "route_duration": pc.field("route_duration").cast(pa.float32()),
    }
)
osrm_table2.schema

idx: uint32
route_distance: float
route_duration: float

In [8]:
feats_v1.schema

idx: uint32
fare_amount: halffloat
passenger_count: uint8
travel_distance: halffloat
trip_dow: uint8
trip_year: uint16
trip_month: uint8
trip_day: uint8
trip_hour: uint8
dropoff_jfk_airport_dist: halffloat
pickup_jfk_airport_dist: halffloat
dropoff_lag_airport_dist: halffloat
pickup_lag_airport_dist: halffloat
dropoff_new_airport_dist: halffloat
pickup_new_airport_dist: halffloat
dropoff_center_dist: halffloat
pickup_center_dist: halffloat
dropoff_manhattan_dist: halffloat
pickup_manhattan_dist: halffloat
dropoff_brooklyn_dist: halffloat
pickup_brooklyn_dist: halffloat
dropoff_queens_dist: halffloat
pickup_queens_dist: halffloat
dropoff_bronx_dist: halffloat
pickup_bronx_dist: halffloat
dropoff_staten_dist: halffloat
pickup_staten_dist: halffloat

In [12]:
res_table = feats_v1.join(osrm_table2, keys="idx", join_type='inner')
res_norm = data.normalize_table(res_table)
batches = res_norm.to_batches(max_chunksize=data.MAX_CHUNK_SIZE)
res_norm = pa.Table.from_batches(batches)
pf.write_feather(res_norm, "data_v2.feather")

In [13]:
print(feats_v1.shape)
print(res_norm.shape)
for col in res_table.column_names:
    if not col.startswith("route_"):
        continue
    cnt_null = pc.sum(pc.is_null(res_table[col]))
    cnt_nan = pc.sum(pc.is_nan(res_table[col]))
    print(f"{col}: nulls={cnt_null}, nans={cnt_nan}")

(54273658, 27)
(54273658, 29)
route_distance: nulls=0, nans=0
route_duration: nulls=0, nans=0
